# ECE 285 Homework 3: Denoising Diffusion Probabilistic Model (DDPM) for Histopathology Image Generation

This notebook implements a **Denoising Diffusion Probabilistic Model (DDPM)** from scratch in PyTorch and trains it on the **Histopathology** dataset (`data/Histopathology`, 5,179 images) using **Apple Silicon MPS (`mps`)** hardware acceleration.

It also produces all artifacts required for the 2-page NeurIPS report:
1. **Hyperparameter settings** (cleanly summarized in a structured table)
2. **Training loss curves** (`outputs/training_loss_curve.png`)
3. **Inception Score (IS) and Fréchet Inception Distance (FID)** computed with **PyTorch-Ignite** (`ignite.metrics.FID` & `ignite.metrics.InceptionScore`, following the PyTorch-Ignite GAN evaluation tutorial)
4. **Examples of generated images** and denoising trajectories (`outputs/generated_samples.png`, `outputs/real_vs_generated.png`, `outputs/denoising_trajectory.png`)

## 1. Environment Setup & MPS Hardware Acceleration

In [ ]:
import os
import math
import json
import copy
import time
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.utils as vutils

import ignite
from ignite.engine import Engine
from ignite.metrics import FID, InceptionScore

# Allow MPS fallback for any unsupported ops if ever encountered
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

print(f"PyTorch     : {torch.__version__}")
print(f"Ignite      : {ignite.__version__}")
print(f"MPS Built   : {torch.backends.mps.is_built()}")
print(f"MPS Avail   : {torch.backends.mps.is_available()}")


def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(f"Active Device: {device}")
    return device


def set_seed(seed: int = 42, device: torch.device | None = None) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device is not None and device.type == "mps":
        torch.mps.manual_seed(seed)
    elif torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = get_device()

## 2. Hyperparameter Configuration

All key hyperparameters for data processing, the U-Net backbone, the DDPM noise schedule, optimization, and PyTorch-Ignite FID/IS evaluation are centralized below.

In [ ]:
@dataclass
class Config:
    # Paths
    data_dir: str = "./data/Histopathology"
    output_dir: str = "./outputs"
    checkpoint_dir: str = "./checkpoints"

    # Reproducibility
    seed: int = 42

    # Data
    image_size: int = 64          # Resized from 256x256 for fast, stable MPS training (can set to 128 if desired)
    in_channels: int = 3
    batch_size: int = 64
    num_workers: int = 0          # 0 is safest and fastest with MPS + Jupyter on macOS

    # U-Net Architecture
    base_channels: int = 64
    channel_mults: tuple = (1, 2, 4, 4)
    num_res_blocks: int = 2
    attn_resolutions: tuple = (16, 8)
    dropout: float = 0.1
    num_heads: int = 4

    # Diffusion Schedule (DDPM)
    timesteps: int = 1000
    beta_schedule: str = "linear" # "linear" or "cosine"
    beta_start: float = 1e-4
    beta_end: float = 0.02

    # Optimization
    epochs: int = 80
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    ema_decay: float = 0.9999

    # Logging & Checkpointing
    sample_interval: int = 10     # Generate sample grids every N epochs
    eval_batch_size: int = 64
    eval_num_samples: int = 1024  # Number of real/fake pairs for FID & IS (set to 5179 for full dataset)
    eval_use_ddim: bool = False   # Set True (e.g. 100 steps) for ~10x faster sampling during evaluation
    ddim_steps: int = 100
    ddim_eta: float = 0.0


cfg = Config()
set_seed(cfg.seed, device)

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)

print("=== Hyperparameter Settings ===")
for k, v in asdict(cfg).items():
    print(f"{k:<20s}: {v}")

with open(Path(cfg.output_dir) / "hyperparameters.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)

## 3. Dataset Loading & Exploration (`data/Histopathology`)

The `data/Histopathology` folder contains 5,179 RGB PNG histopathology patches (`256 x 256`).
We define a custom `HistopathologyDataset` that loads all `.png` files directly from the folder, ignores hidden system files (e.g., `.DS_Store`), applies random horizontal/vertical flips during training (histopathology tissues are rotation/reflection invariant), and normalizes pixel values to `[-1, 1]`.

In [ ]:
class HistopathologyDataset(Dataset):
    def __init__(self, root_dir: str, transform=None):
        self.root_dir = Path(root_dir)
        self.image_paths = sorted(
            [p for p in self.root_dir.iterdir() if p.is_file() and p.suffix.lower() == ".png"]
        )
        if len(self.image_paths) == 0:
            raise RuntimeError(f"No PNG images found in {self.root_dir.resolve()}")
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img


train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # [-1, 1]
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # [-1, 1]
])

train_dataset = HistopathologyDataset(cfg.data_dir, transform=train_transform)
eval_full_dataset = HistopathologyDataset(cfg.data_dir, transform=eval_transform)

num_eval = min(cfg.eval_num_samples, len(eval_full_dataset))
eval_indices = torch.arange(num_eval)
eval_dataset = Subset(eval_full_dataset, eval_indices)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=False,
    drop_last=True,
)

eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=False,
    drop_last=False,
)

print(f"Total Histopathology images : {len(train_dataset)}")
print(f"Training batches / epoch    : {len(train_dataloader)}")
print(f"Evaluation images (FID/IS)  : {len(eval_dataset)}")

# Visualize a batch of real training images
real_batch = next(iter(train_dataloader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Sample Real Histopathology Images (64x64)")
grid = vutils.make_grid(real_batch[:64], nrow=8, padding=2, normalize=True, value_range=(-1, 1))
plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "real_samples.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Denoising U-Net Architecture

We implement a time-conditioned U-Net (`UNet`) with:
- Sinusoidal timestep embeddings projected through a two-layer SiLU MLP
- Residual blocks (`ResBlock`) with `GroupNorm`, `SiLU`, time-embedding injection, and dropout
- Multi-head self-attention (`SelfAttention2d`) at lower spatial resolutions (`16x16` and `8x8`)
- Skip connections across all encoder and decoder levels

All operations use `float32` tensors for full compatibility with Apple Silicon `mps`.

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, time: torch.Tensor) -> torch.Tensor:
        device = time.device
        half_dim = self.dim // 2
        emb_scale = math.log(10000.0) / max(half_dim - 1, 1)
        freqs = torch.exp(torch.arange(half_dim, device=device, dtype=torch.float32) * -emb_scale)
        args = time.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat((args.sin(), args.cos()), dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


def _num_groups(channels: int, max_groups: int = 32) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


class ResBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, time_dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(_num_groups(in_channels), in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_channels),
        )

        self.norm2 = nn.GroupNorm(_num_groups(out_channels), out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.shortcut(x)


class SelfAttention2d(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        assert channels % num_heads == 0, f"channels ({channels}) must be divisible by num_heads ({num_heads})"
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.norm = nn.GroupNorm(_num_groups(channels), channels)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        norm_x = self.norm(x)
        qkv = self.qkv(norm_x)  # (B, 3C, H, W)
        q, k, v = qkv.chunk(3, dim=1)

        # Reshape to (B, heads, HW, head_dim)
        q = q.view(b, self.num_heads, self.head_dim, h * w).transpose(2, 3)
        k = k.view(b, self.num_heads, self.head_dim, h * w)
        v = v.view(b, self.num_heads, self.head_dim, h * w).transpose(2, 3)

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k) * scale  # (B, heads, HW, HW)
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)  # (B, heads, HW, head_dim)
        out = out.transpose(2, 3).contiguous().view(b, c, h, w)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        return self.conv(x)


class UNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        base_channels: int = 64,
        channel_mults: tuple = (1, 2, 4, 4),
        num_res_blocks: int = 2,
        image_size: int = 64,
        attn_resolutions: tuple = (16, 8),
        dropout: float = 0.1,
        num_heads: int = 4,
    ):
        super().__init__()
        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.init_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # Downsampling path
        self.down_blocks = nn.ModuleList()
        ch = base_channels
        curr_res = image_size
        skip_channels = [ch]
        num_levels = len(channel_mults)

        for level, mult in enumerate(channel_mults):
            out_ch = base_channels * mult
            for _ in range(num_res_blocks):
                layers = nn.ModuleList([ResBlock(ch, out_ch, time_dim, dropout)])
                if curr_res in attn_resolutions:
                    layers.append(SelfAttention2d(out_ch, num_heads=num_heads))
                self.down_blocks.append(layers)
                ch = out_ch
                skip_channels.append(ch)
            if level != num_levels - 1:
                self.down_blocks.append(nn.ModuleList([Downsample(ch)]))
                skip_channels.append(ch)
                curr_res //= 2

        # Bottleneck
        self.mid_block1 = ResBlock(ch, ch, time_dim, dropout)
        self.mid_attn = SelfAttention2d(ch, num_heads=num_heads)
        self.mid_block2 = ResBlock(ch, ch, time_dim, dropout)

        # Upsampling path
        self.up_blocks = nn.ModuleList()
        for level, mult in reversed(list(enumerate(channel_mults))):
            out_ch = base_channels * mult
            for i in range(num_res_blocks + 1):
                skip_ch = skip_channels.pop()
                layers = nn.ModuleList([ResBlock(ch + skip_ch, out_ch, time_dim, dropout)])
                if curr_res in attn_resolutions:
                    layers.append(SelfAttention2d(out_ch, num_heads=num_heads))
                if level != 0 and i == num_res_blocks:
                    layers.append(Upsample(out_ch))
                    curr_res *= 2
                self.up_blocks.append(layers)
                ch = out_ch

        self.out_norm = nn.GroupNorm(_num_groups(ch), ch)
        self.out_conv = nn.Conv2d(ch, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(t)
        h = self.init_conv(x)
        hs = [h]

        for block in self.down_blocks:
            if isinstance(block[0], Downsample):
                h = block[0](h)
            else:
                h = block[0](h, t_emb)
                if len(block) > 1:
                    h = block[1](h)
            hs.append(h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        for block in self.up_blocks:
            skip = hs.pop()
            h = torch.cat([h, skip], dim=1)
            h = block[0](h, t_emb)
            for layer in block[1:]:
                h = layer(h)

        return self.out_conv(F.silu(self.out_norm(h)))

## 5. DDPM Forward/Reverse Process & Exponential Moving Average (EMA)

We implement `GaussianDiffusion` following Ho et al. (2020):
- Forward process: $q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t)\mathbf{I})$
- Training objective: $\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\big[\|\epsilon - \epsilon_\theta(x_t, t)\|_2^2\big]$
- Reverse process: standard 1,000-step DDPM ancestral sampling with clamped $\hat{x}_0$ prediction, plus optional DDIM sampling (`sample_ddim`) and trajectory visualization (`sample_trajectory`).
- EMA wrapper (`EMA`) maintains smoothed model weights (`decay = 0.9999`), which substantially improves sample quality in diffusion models.

In [ ]:
def linear_beta_schedule(timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)


def cosine_beta_schedule(timesteps: int, s: float = 0.008) -> torch.Tensor:
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float32)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1.0 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)


def extract(a: torch.Tensor, t: torch.Tensor, x_shape: tuple) -> torch.Tensor:
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


class GaussianDiffusion(nn.Module):
    def __init__(
        self,
        model: nn.Module,
        image_size: int = 64,
        in_channels: int = 3,
        timesteps: int = 1000,
        beta_schedule: str = "linear",
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
    ):
        super().__init__()
        self.model = model
        self.image_size = image_size
        self.in_channels = in_channels
        self.timesteps = timesteps

        if beta_schedule == "linear":
            betas = linear_beta_schedule(timesteps, beta_start, beta_end)
        elif beta_schedule == "cosine":
            betas = cosine_beta_schedule(timesteps)
        else:
            raise ValueError(f"Unknown beta_schedule: {beta_schedule}")

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("alphas_cumprod_prev", alphas_cumprod_prev)

        # Forward process q(x_t | x_0)
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))

        # Reverse process q(x_{t-1} | x_t, x_0)
        self.register_buffer("sqrt_recip_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod))
        self.register_buffer("sqrt_recipm1_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod - 1.0))

        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.register_buffer("posterior_variance", posterior_variance)
        self.register_buffer("posterior_log_variance_clipped", torch.log(torch.clamp(posterior_variance, min=1e-20)))
        self.register_buffer(
            "posterior_mean_coef1",
            betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod),
        )
        self.register_buffer(
            "posterior_mean_coef2",
            (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod),
        )

    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor, noise: torch.Tensor | None = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x_start)
        return (
            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise
        )

    def training_loss(self, x_start: torch.Tensor) -> torch.Tensor:
        b = x_start.shape[0]
        t = torch.randint(0, self.timesteps, (b,), device=x_start.device, dtype=torch.long)
        noise = torch.randn_like(x_start)
        x_noisy = self.q_sample(x_start, t, noise=noise)
        pred_noise = self.model(x_noisy, t)
        return F.mse_loss(pred_noise, noise)

    def predict_start_from_noise(self, x_t: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        return (
            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t
            - extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
        )

    @torch.no_grad()
    def p_sample(self, x_t: torch.Tensor, t_idx: int) -> torch.Tensor:
        b = x_t.shape[0]
        t = torch.full((b,), t_idx, device=x_t.device, dtype=torch.long)
        pred_noise = self.model(x_t, t)
        x_start = self.predict_start_from_noise(x_t, t, pred_noise)
        x_start = torch.clamp(x_start, -1.0, 1.0)

        model_mean = (
            extract(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        if t_idx == 0:
            return model_mean
        posterior_var = extract(self.posterior_variance, t, x_t.shape)
        noise = torch.randn_like(x_t)
        return model_mean + torch.sqrt(posterior_var) * noise

    @torch.no_grad()
    def sample(
        self,
        batch_size: int,
        noise: torch.Tensor | None = None,
        show_progress: bool = False,
        return_trajectory: bool = False,
        trajectory_steps: tuple = (999, 800, 600, 400, 200, 100, 50, 0),
    ):
        dev = self.betas.device
        x = noise.to(dev) if noise is not None else torch.randn(
            batch_size, self.in_channels, self.image_size, self.image_size, device=dev
        )
        traj = {}
        iterator = reversed(range(self.timesteps))
        if show_progress:
            iterator = tqdm(list(iterator), desc="DDPM Sampling", leave=False)

        for t_idx in iterator:
            x = self.p_sample(x, t_idx)
            if return_trajectory and t_idx in trajectory_steps:
                traj[t_idx] = torch.clamp(x, -1.0, 1.0).detach().cpu()

        x = torch.clamp(x, -1.0, 1.0)
        if return_trajectory:
            return x, traj
        return x

    @torch.no_grad()
    def sample_ddim(
        self,
        batch_size: int,
        ddim_steps: int = 100,
        eta: float = 0.0,
        noise: torch.Tensor | None = None,
        show_progress: bool = False,
    ) -> torch.Tensor:
        dev = self.betas.device
        x = noise.to(dev) if noise is not None else torch.randn(
            batch_size, self.in_channels, self.image_size, self.image_size, device=dev
        )
        step_indices = torch.linspace(0, self.timesteps - 1, ddim_steps, dtype=torch.long).tolist()
        step_indices = list(reversed(step_indices))

        iterator = range(len(step_indices))
        if show_progress:
            iterator = tqdm(list(iterator), desc=f"DDIM ({ddim_steps} steps)", leave=False)

        for i in iterator:
            t_cur = step_indices[i]
            t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else -1

            t_tensor = torch.full((x.shape[0],), t_cur, device=dev, dtype=torch.long)
            pred_noise = self.model(x, t_tensor)

            alpha_bar_t = self.alphas_cumprod[t_cur]
            alpha_bar_prev = self.alphas_cumprod[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=dev)

            x_0_pred = (x - torch.sqrt(1.0 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)
            x_0_pred = torch.clamp(x_0_pred, -1.0, 1.0)

            sigma = (
                eta
                * torch.sqrt((1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t))
                * torch.sqrt(1.0 - alpha_bar_t / alpha_bar_prev)
            )
            dir_xt = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma ** 2, min=0.0)) * pred_noise
            z = torch.randn_like(x) if t_prev >= 0 and eta > 0 else 0.0
            x = torch.sqrt(alpha_bar_prev) * x_0_pred + dir_xt + sigma * z

        return torch.clamp(x, -1.0, 1.0)


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module, step: int | None = None) -> None:
        decay = self.decay
        if step is not None:
            decay = min(self.decay, (1.0 + step) / (10.0 + step))
        for ema_p, p in zip(self.ema_model.parameters(), model.parameters()):
            ema_p.data.mul_(decay).add_(p.data, alpha=1.0 - decay)
        for ema_b, b in zip(self.ema_model.buffers(), model.buffers()):
            ema_b.copy_(b)

## 6. Model Initialization & Training on MPS

We instantiate the `UNet` and `GaussianDiffusion` on `device` (`mps`), optimize with `AdamW` (`lr = 2e-4`) and a cosine learning rate scheduler, clip gradient norms to `1.0`, and maintain an EMA copy of the diffusion model. Checkpoints and sample previews are saved periodically.

In [ ]:
unet = UNet(
    in_channels=cfg.in_channels,
    base_channels=cfg.base_channels,
    channel_mults=cfg.channel_mults,
    num_res_blocks=cfg.num_res_blocks,
    image_size=cfg.image_size,
    attn_resolutions=cfg.attn_resolutions,
    dropout=cfg.dropout,
    num_heads=cfg.num_heads,
).to(device)

diffusion = GaussianDiffusion(
    model=unet,
    image_size=cfg.image_size,
    in_channels=cfg.in_channels,
    timesteps=cfg.timesteps,
    beta_schedule=cfg.beta_schedule,
    beta_start=cfg.beta_start,
    beta_end=cfg.beta_end,
).to(device)

num_params = sum(p.numel() for p in unet.parameters())
print(f"U-Net Trainable Parameters: {num_params:,} ({num_params / 1e6:.2f} M)")

ema = EMA(diffusion, decay=cfg.ema_decay)
optimizer = torch.optim.AdamW(diffusion.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

fixed_noise = torch.randn(64, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)

step_losses = []
epoch_losses = []
global_step = 0
start_time = time.time()

for epoch in range(1, cfg.epochs + 1):
    diffusion.train()
    running_loss = 0.0
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch:03d}/{cfg.epochs:03d}", leave=False)

    for real_imgs in pbar:
        real_imgs = real_imgs.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = diffusion.training_loss(real_imgs)
        loss.backward()
        if cfg.grad_clip > 0:
            nn.utils.clip_grad_norm_(diffusion.parameters(), cfg.grad_clip)
        optimizer.step()

        global_step += 1
        ema.update(diffusion, step=global_step)

        loss_val = float(loss.item())
        step_losses.append(loss_val)
        running_loss += loss_val
        pbar.set_postfix({"loss": f"{loss_val:.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.2e}"})

    scheduler.step()
    avg_epoch_loss = running_loss / len(train_dataloader)
    epoch_losses.append(avg_epoch_loss)
    elapsed = time.time() - start_time
    print(f"Epoch [{epoch:03d}/{cfg.epochs:03d}] | Avg MSE Loss: {avg_epoch_loss:.5f} | Elapsed: {elapsed / 60:.1f} min")

    # Save sample preview and checkpoint periodically
    if epoch % cfg.sample_interval == 0 or epoch == cfg.epochs:
        ema.ema_model.eval()
        with torch.no_grad():
            if cfg.eval_use_ddim:
                samples = ema.ema_model.sample_ddim(
                    batch_size=64, ddim_steps=cfg.ddim_steps, eta=cfg.ddim_eta, noise=fixed_noise
                )
            else:
                samples = ema.ema_model.sample(batch_size=64, noise=fixed_noise)

        grid = vutils.make_grid(samples.cpu(), nrow=8, padding=2, normalize=True, value_range=(-1, 1))
        vutils.save_image(grid, Path(cfg.output_dir) / f"samples_epoch_{epoch:03d}.png")

        ckpt_path = Path(cfg.checkpoint_dir) / "ddpm_histopathology_latest.pt"
        torch.save({
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": diffusion.state_dict(),
            "ema_state_dict": ema.ema_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "step_losses": step_losses,
            "epoch_losses": epoch_losses,
            "config": asdict(cfg),
        }, ckpt_path)

        if device.type == "mps":
            torch.mps.empty_cache()

final_ckpt_path = Path(cfg.checkpoint_dir) / "ddpm_histopathology_final.pt"
torch.save({
    "epoch": cfg.epochs,
    "global_step": global_step,
    "model_state_dict": diffusion.state_dict(),
    "ema_state_dict": ema.ema_model.state_dict(),
    "step_losses": step_losses,
    "epoch_losses": epoch_losses,
    "config": asdict(cfg),
}, final_ckpt_path)
print(f"Training completed in {(time.time() - start_time) / 60:.2f} min. Final checkpoint saved to {final_ckpt_path}")

## 7. Training Loss Curves

We plot both the per-iteration MSE loss (with an exponential moving average curve for readability) and the per-epoch average MSE loss. The high-resolution figure is saved to `outputs/training_loss_curve.png` for inclusion in the NeurIPS report.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: Iteration-level loss + smoothed curve
steps_arr = np.arange(1, len(step_losses) + 1)
axes[0].plot(steps_arr, step_losses, color="steelblue", alpha=0.25, linewidth=0.8, label="Raw Iteration Loss")

if len(step_losses) >= 20:
    window = max(10, len(step_losses) // 100)
    kernel = np.ones(window) / window
    smoothed = np.convolve(step_losses, kernel, mode="valid")
    axes[0].plot(
        np.arange(window, len(step_losses) + 1),
        smoothed,
        color="navy",
        linewidth=1.8,
        label=f"Moving Avg (window={window})",
    )

axes[0].set_title("DDPM Training Loss vs. Iterations")
axes[0].set_xlabel("Training Iteration")
axes[0].set_ylabel("MSE Loss")
axes[0].grid(True, linestyle="--", alpha=0.4)
axes[0].legend()

# Right: Epoch-level average loss
epochs_arr = np.arange(1, len(epoch_losses) + 1)
axes[1].plot(epochs_arr, epoch_losses, marker="o", markersize=3.5, color="crimson", linewidth=1.8, label="Epoch Avg MSE")
axes[1].set_title("DDPM Epoch Average Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Average MSE Loss")
axes[1].grid(True, linestyle="--", alpha=0.4)
axes[1].legend()

plt.tight_layout()
loss_fig_path = Path(cfg.output_dir) / "training_loss_curve.png"
plt.savefig(loss_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved training loss curve to {loss_fig_path}")

## 8. Quantitative Evaluation: FID & Inception Score (PyTorch-Ignite)

Following the PyTorch-Ignite evaluation code (`https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/`), we:
1. Resize both real and generated images from `64 x 64` to `299 x 299` using `PIL.Image.BILINEAR` (via `interpolate()`), properly mapping `[-1, 1]` tensors to `[0, 1]` before `ToPILImage()` so negative pixel values are not clipped.
2. Instantiate `FID(device=device)` and `InceptionScore(device=device, output_transform=lambda x: x[0])`. Note that PyTorch-Ignite `>= 0.5` automatically uses `float32` accumulators on `mps` (since Apple MPS does not support `float64`).
3. Run an Ignite `Engine(evaluation_step)` over `eval_dataloader` and record the final FID and IS scores.

We also cache the generated samples from the evaluation run so they can be directly visualized in the next section.

In [ ]:
# Optional: Load checkpoint if running evaluation standalone
# ckpt = torch.load(Path(cfg.checkpoint_dir) / "ddpm_histopathology_final.pt", map_location=device)
# ema.ema_model.load_state_dict(ckpt["ema_state_dict"])

eval_model = ema.ema_model.to(device).eval()

fid_metric = FID(device=device)
is_metric = InceptionScore(device=device, output_transform=lambda x: x[0])


def interpolate(batch: torch.Tensor) -> torch.Tensor:
    """
    Resize a batch of [-1, 1] tensors to (299, 299) using PIL.Image.BILINEAR,
    consistent with the PyTorch-Ignite FID/IS tutorial.
    """
    arr = []
    for img in batch:
        img_01 = (img.detach().cpu().clamp(-1.0, 1.0) + 1.0) / 2.0
        pil_img = transforms.ToPILImage()(img_01)
        resized_img = pil_img.resize((299, 299), Image.BILINEAR)
        arr.append(transforms.ToTensor()(resized_img))
    return torch.stack(arr)


generated_eval_batches = []
real_eval_batches = []


def evaluation_step(engine, real_batch):
    cur_bs = real_batch.shape[0]
    with torch.no_grad():
        if cfg.eval_use_ddim:
            fake_batch = eval_model.sample_ddim(
                batch_size=cur_bs, ddim_steps=cfg.ddim_steps, eta=cfg.ddim_eta
            )
        else:
            fake_batch = eval_model.sample(batch_size=cur_bs)

    generated_eval_batches.append(fake_batch.cpu())
    real_eval_batches.append(real_batch.cpu())

    fake_299 = interpolate(fake_batch)
    real_299 = interpolate(real_batch)
    return fake_299, real_299


evaluator = Engine(evaluation_step)
fid_metric.attach(evaluator, "fid")
is_metric.attach(evaluator, "is")

print(f"Running PyTorch-Ignite FID & IS evaluation on {len(eval_dataset)} samples (device={device})...")
eval_start = time.time()
evaluator.run(eval_dataloader, max_epochs=1)
eval_metrics = evaluator.state.metrics

fid_score_val = float(eval_metrics["fid"])
is_score_val = float(eval_metrics["is"])
eval_elapsed = time.time() - eval_start

print(f"Evaluation completed in {eval_elapsed / 60:.2f} min")
print(f"  * FID Score       : {fid_score_val:.4f} (lower is better)")
print(f"  * Inception Score : {is_score_val:.4f} (higher is better)")

all_fake_images = torch.cat(generated_eval_batches, dim=0)
all_real_images = torch.cat(real_eval_batches, dim=0)

results_summary = {
    "num_eval_samples": len(eval_dataset),
    "sampler": f"DDIM ({cfg.ddim_steps} steps)" if cfg.eval_use_ddim else f"DDPM ({cfg.timesteps} steps)",
    "fid_score": fid_score_val,
    "inception_score": is_score_val,
    "final_epoch_loss": epoch_losses[-1] if epoch_losses else None,
    "best_epoch_loss": min(epoch_losses) if epoch_losses else None,
    "num_parameters": num_params,
}
with open(Path(cfg.output_dir) / "metrics_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

## 9. Generated Histopathology Images & Denoising Trajectory

We visualize:
1. A side-by-side comparison of **Real Histopathology Images** vs. **DDPM Generated Images** (`outputs/real_vs_generated.png`).
2. A standalone `8 x 8` grid of **Generated Histopathology Images** (`outputs/generated_samples.png`).
3. The progressive reverse diffusion trajectory from pure Gaussian noise ($t=999$) to clean histopathology patches ($t=0$) (`outputs/denoising_trajectory.png`).

In [ ]:
# 1. Real vs. Generated Comparison Grid
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

real_grid = vutils.make_grid(all_real_images[:64], nrow=8, padding=3, normalize=True, value_range=(-1, 1))
fake_grid = vutils.make_grid(all_fake_images[:64], nrow=8, padding=3, normalize=True, value_range=(-1, 1))

axes[0].imshow(np.transpose(real_grid.numpy(), (1, 2, 0)))
axes[0].set_title("Real Histopathology Images", fontsize=14)
axes[0].axis("off")

axes[1].imshow(np.transpose(fake_grid.numpy(), (1, 2, 0)))
axes[1].set_title(f"DDPM Generated Images (FID={fid_score_val:.2f}, IS={is_score_val:.2f})", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "real_vs_generated.png", dpi=300, bbox_inches="tight")
plt.show()

# 2. Standalone Generated Samples Grid
plt.figure(figsize=(8, 8))
plt.imshow(np.transpose(fake_grid.numpy(), (1, 2, 0)))
plt.title("Generated Histopathology Samples (DDPM)", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "generated_samples.png", dpi=300, bbox_inches="tight")
plt.show()

# 3. Denoising Trajectory Visualization
traj_steps = (999, 800, 600, 400, 200, 100, 50, 0)
with torch.no_grad():
    _, trajectory = eval_model.sample(
        batch_size=4,
        show_progress=True,
        return_trajectory=True,
        trajectory_steps=traj_steps,
    )

fig, axes = plt.subplots(4, len(traj_steps), figsize=(2.0 * len(traj_steps), 8.2))
for col, t_step in enumerate(traj_steps):
    imgs_t = (trajectory[t_step].clamp(-1.0, 1.0) + 1.0) / 2.0
    for row in range(4):
        ax = axes[row, col]
        ax.imshow(np.transpose(imgs_t[row].numpy(), (1, 2, 0)))
        ax.axis("off")
        if row == 0:
            ax.set_title(f"t = {t_step}", fontsize=11)

plt.suptitle(r"Reverse Diffusion Denoising Process ($x_T \rightarrow x_0$)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "denoising_trajectory.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Report Summary

In [ ]:
print("=" * 60)
print("Summary for 2-Page NeurIPS Report")
print("=" * 60)
print(f"Dataset              : Histopathology ({len(train_dataset)} images, resized to {cfg.image_size}x{cfg.image_size})")
print(f"Hardware Accelerator : {device}")
print(f"Model                : Time-Conditioned U-Net ({num_params / 1e6:.2f}M params)")
print(f"Diffusion Schedule   : {cfg.beta_schedule}, T={cfg.timesteps}, beta=[{cfg.beta_start}, {cfg.beta_end}]")
print(f"Optimizer            : AdamW (lr={cfg.lr}, wd={cfg.weight_decay}), CosineAnnealingLR, EMA={cfg.ema_decay}")
print(f"Batch Size / Epochs  : {cfg.batch_size} / {cfg.epochs}")
if epoch_losses:
    print(f"Final Epoch MSE Loss : {epoch_losses[-1]:.5f} (Best: {min(epoch_losses):.5f})")
print(f"FID Score (Ignite)   : {fid_score_val:.4f}")
print(f"Inception Score      : {is_score_val:.4f}")
print("Saved Artifacts:")
for p in sorted(Path(cfg.output_dir).glob("*")):
    print(f"  - {p}")
print("=" * 60)